#### Section 1 — Environment setup

In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [4]:
assert GROQ_API_KEY,   "GROQ_API_KEY missing from .env"
assert TAVILY_API_KEY, "TAVILY_API_KEY missing from .env"

#### Section 2 — LLM integration

In [5]:
from langchain_groq import ChatGroq

In [6]:
llm = ChatGroq(
    model = "llama-3.1-8b-instant",
    temperature = "0.3",
    api_key = GROQ_API_KEY
)

In [9]:
llm.model_name

'llama-3.1-8b-instant'

In [10]:
# invoke(): send one prompt, get one complete response back
response = llm.invoke("In one sentence, what does a Data Engineer do?")
print(f"Content: {response.content}")
print(f"Metadata: {response.response_metadata}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}

In [ ]:
# stream(): tokens arrive in real time, great for chat UIs
print("Streaming response:\n")

full_response: str = ""
for chunk in llm.stream("Give me 3 bullet points to strengthen a weak resume summary."):
    print(chunk.content, end="", flush=True)
    full_response += chunk.content

print("\n\nStream complete. Total chars: ", len(full_response))

Streaming response:

Here are three bullet points to help strengthen a weak resume summary:

• **Quantify achievements**: Instead of making generic claims, use specific numbers and metrics to demonstrate your accomplishments. For example, "Increased sales by 25% in 6 months" or "Managed a team of 10 people, resulting in a 30% increase in productivity."

• **Highlight relevant skills and certifications**: Emphasize the skills and certifications that are most relevant to the job you're applying for. This will help you stand out from other applicants and show that you have the necessary qualifications for the position. For example, "Certified in project management (PMP) with 5 years of experience in leading cross-functional teams."

• **Tailor the summary to the job**: Customize your resume summary to match the requirements and qualifications listed in the job posting. This will show that you've taken the time to understand the employer's needs and that you have the skills and experience 

In [10]:
# batch(): process multiple inputs in parallel
job_roles = [
    "What are the top 5 skills for a Data Engineer?",
    "What are the top 5 skills for a Frontend Developer?",
    "What are the top 5 skills for a DevOps Engineer?"
]

responses = llm.batch(job_roles)

for role_prompt, response in zip(job_roles, responses):
    role = role_prompt.split("for a")[1].strip().rstrip("?")
    print(f"\n--- {role} ---")
    print(response.content[:200], "...")


--- Data Engineer ---
Based on industry trends and job requirements, here are the top 5 skills for a Data Engineer:

1. **Programming Skills**: Proficiency in programming languages such as:
	* Python: widely used in data e ...

--- Frontend Developer ---
Here are the top 5 skills for a Frontend Developer:

1. **HTML/CSS**: A solid understanding of HTML (Hypertext Markup Language) and CSS (Cascading Style Sheets) is essential for building the structure ...

--- DevOps Engineer ---
As a DevOps Engineer, the top 5 skills required are:

1. **Automation and Scripting**: Proficiency in scripting languages such as Python, PowerShell, or Bash is essential for automating tasks, deployi ...


#### Section 3 — Prompt engineering

In [11]:
# Three types of messages
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# System message: set the llm role
system = SystemMessage(content="""
You are an expert career coach and resume analyst.
Your job is to analyze resumes and give sharp, actionable feedback.
Always be concise. Use bullet points. Never be vague.
""")

# Human message: user's input
human = HumanMessage(content="""
Here is my resume summary:
'Hardworking professional with good communication skills and a passion for technology.'

What is wrong with this summary and how do I fix it?
""")

response = llm.invoke([system, human])
print(response.content)

**Weaknesses:**

* Lack of specificity and uniqueness
* No clear career goals or value proposition
* Overused and generic phrases

**Fix:**

* Quantify your skills and achievements (e.g., "Proven track record of improving project efficiency by 25% through effective communication")
* Highlight your unique strengths and qualifications (e.g., "Certified in cloud computing with 3+ years of experience")
* Tailor your summary to your target industry or job role (e.g., "Results-driven IT professional with a passion for cybersecurity")

**Example:**

"Results-driven IT professional with 3+ years of experience in cloud computing. Proven track record of improving project efficiency by 25% through effective communication. Certified in cloud security and committed to staying at the forefront of industry trends."


In [12]:
# Let's give fake memory to llm by inserting an AIMessage

conversation = [
    SystemMessage(content="You are a strict resume coach. Be blunt and direct."),
    HumanMessage(content="Here is my resume summary: 'Experienced developer.'"),
    AIMessage(content="This summary is too vague. Add your years of experience, your main tech stack, and one measurable achievement."),
    HumanMessage(content="Okay. How about: 'Python developer with 3 years building REST APIs.'"),
]

response = llm.invoke(conversation)
print(response.content)

That's a slight improvement, but still not specific enough. Consider adding a quantifiable achievement, such as:

- 'Highly skilled Python developer with 3 years of experience building scalable REST APIs, resulting in a 30% increase in application performance.'
- 'Results-driven Python developer with 3 years of experience designing and implementing REST APIs, achieving a 95% uptime rate.'
- 'Experienced Python developer with 3 years of experience developing and deploying REST APIs, with a focus on security and compliance, resulting in a 25% reduction in security breaches.'

Remember, the goal is to showcase your skills and achievements in a concise and impactful way.


#### Section 4 — Tools

In [13]:
# @tool: turn any Python function into a LangChain tool
from langchain_core.tools import tool

@tool
def analyze_skill_gap(resume_skills: str, job_required_skills: str) -> str:
    """
    Compares skills listed in a resume against skills required in a job description.
    Returns missing skills and matching skills as a structured summary.
    
    Args:
        resume_skills: comma-separated skills found in the resume
        job_required_skills: comma-separated skills required by the job
    """
    resume_set = {s.strip().lower() for s in resume_skills.split(",")}
    job_set    = {s.strip().lower() for s in job_required_skills.split(",")}

    matched = resume_set & job_set
    missing = job_set - resume_set

    return (
        f"Matched skills  ({len(matched)}): {', '.join(sorted(matched)) or 'none'}\n"
        f"Missing skills  ({len(missing)}): {', '.join(sorted(missing)) or 'none'}\n"
        f"Match score: {round(len(matched) / max(len(job_set), 1) * 100)}%"
    )

# Test
result = analyze_skill_gap.invoke({
    "resume_skills": "Python, SQL, Docker, Communication, Excel",
    "job_required_skills": "Python, SQL, Spark, Kafka, Airflow, Docker"
})
print(result)

Matched skills  (3): docker, python, sql
Missing skills  (3): airflow, kafka, spark
Match score: 50%


In [14]:
# Tavily: gives the agent live web search ability
from langchain_community.tools.tavily_search import TavilySearchResults

web_search = TavilySearchResults(
    max_result = 3,
    api_key = TAVILY_API_KEY
)

results = web_search.invoke("top skills required for Data Engineer 2026")

for index, response in enumerate(results, 1):
    print(f"\n[{index}] {response['url']}")
    print(response['content'][:200], "...")

C:\Users\raoru.DELL\AppData\Local\Temp\ipykernel_8244\1001786348.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(



[1] https://iabac.org/blog/what-are-the-top-data-engineer-skills
This improves reliability and reduces production failures.

4. Cloud Platforms: AWS, Azure, GCP

The majority of data systems in 2026 are cloud-based.

Cloud services are used by businesses for pipeli ...

[2] https://www.refontelearning.com/blog/data-engineering-in-2026-trends-skills-and-how-to-thrive-in-a-data-driven-world
For someone entering the field, it’s worth noting that soft boundaries between data roles are an ongoing trend. A solid data engineer in 2026 has a T-shaped skill profile: deep expertise in building d ...

[3] https://www.dataquest.io/blog/data-engineering-skills/
Code reviews, pair programming, and cross-functional project work all require collaboration skills that go beyond technical ability.

### Adaptability

The data engineering tooling landscape changes c ...

[4] https://bix-tech.com/data-engineering-in-2026-skills-that-truly-differentiate-senior-professionals/
Ownership: end-to-end responsibi

In [15]:
#  Force the LLM to return strict JSON using Pydantic
from pydantic import BaseModel, Field
from typing import List

class ResumeAnalysis(BaseModel):
    """Structured output schema for resume analysis results."""
    match_score: int = Field(
        description="Overall match percentage between resume and job description (0-100)"
    )
    matched_skills: List[str] = Field(
        description="Skills present in both resume and job description"
    )
    missing_skills: List[str] = Field(
        description="Skills required by the job but absent from the resume"
    )
    rewritten_summary: str = Field(
        description="An improved, stronger resume summary tailored to the job"
    )
    top_recommendations: List[str] = Field(
        description="3 to 5 specific actionable improvements for this resume"
    )

structured_llm = llm.with_structured_output(ResumeAnalysis)

test_result = structured_llm.invoke("""
Analyze this resume against the job description and return structured feedback.

Resume summary: Python developer with 3 years experience building REST APIs using FastAPI and PostgreSQL.

Job description: Looking for a Senior Backend Engineer. Must have: Python, FastAPI, PostgreSQL, Docker, Kubernetes, CI/CD, AWS.
""")

print("Type        :", type(test_result))
print("Match score :", test_result.match_score, "%")
print("Matched     :", test_result.matched_skills)
print("Missing     :", test_result.missing_skills)
print("Rewrite     :", test_result.rewritten_summary)
print("Tips        :")
for tip in test_result.top_recommendations:
    print("  -", tip)

Type        : <class '__main__.ResumeAnalysis'>
Match score : 80 %
Matched     : ['Python', 'FastAPI', 'PostgreSQL']
Missing     : ['Docker', 'Kubernetes', 'CI/CD', 'AWS']
Rewrite     : Highly skilled Senior Backend Engineer with 3 years experience building scalable REST APIs using FastAPI and PostgreSQL, seeking to leverage expertise in cloud infrastructure and DevOps tools to drive business growth.
Tips        :
  - Gain experience with Docker and Kubernetes to improve deployment efficiency
  - Develop expertise in CI/CD pipelines to streamline development and testing
  - Explore AWS services to enhance cloud infrastructure and scalability


#### Section 5 — React agent

In [16]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage

# 1. System message: defines the agent's role and personality
SYSTEM_PROMPT = SystemMessage(content="""
You are an expert career coach and resume analyst called ResumeCoach.

You have access to these tools:
- analyze_skill_gap : use this when the user provides BOTH a resume AND a job description
- web_search        : use this to research a company, role, or industry trends
- structured_llm    : this is your internal analysis brain (do not call directly)

Your coaching rules:
1. Always be specific and actionable — never give vague advice
2. When you see a resume + job description, ALWAYS call analyze_skill_gap first
3. If the user asks about a specific company, ALWAYS call web_search
4. Keep replies concise — use bullet points wherever possible
5. End every reply by asking if the user wants to refine anything
""")

# 2. Bundle tools into list
tools = [analyze_skill_gap, web_search]

# 3. Create the agent 
agent = create_react_agent(
    llm,
    tools,
    prompt = SYSTEM_PROMPT
)

print("React agent ready")
print(" Tools bound: ", [t.name for t in tools])

React agent ready
 Tools bound:  ['analyze_skill_gap', 'tavily_search_results_json']


C:\Users\raoru.DELL\AppData\Local\Temp\ipykernel_8244\1580048835.py:25: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [17]:
# Test the agent with a real resume + job description
from langchain_core.messages import HumanMessage

SAMPLE_RESUME = """
Skills: Python, FastAPI, PostgreSQL, Docker, Git
Experience: 2 years as a Backend Developer building REST APIs
Education: B.Tech Computer Science
"""

SAMPLE_JD = """ 
Role: Senior Data Engineer
Required: Python, Apache Spark, Kafka, Airflow, AWS, Docker, SQL
Nice to have: Kubernetes, dbt, Terraform
"""

HUMAN_PROMPT = HumanMessage(content=f"""
Please analyze my resume against this job description.

MY RESUME:
{SAMPLE_RESUME}

JOB DESCRIPTION:
{SAMPLE_JD}
""")

# invoke the agent 
response = agent.invoke({"messages": [HUMAN_PROMPT]})

# last message in the list is always final answer
final_reponse = response["messages"][-1].content

print("Response:\n", final_reponse)

Response:
 It looks like you have a good foundation in Python and Docker, but there are some key skills missing from your resume that are required for the Senior Data Engineer role. I would recommend focusing on learning Apache Spark, Kafka, Airflow, AWS, and SQL to strengthen your application.

Would you like to refine anything?


In [18]:
# Inspect the full reasoning chain, see each step of agent took (tool calls, tool results, reasoning)
print("Full agent reasoning chain:\n")
print("-" * 50)

for i, msg in enumerate(response["messages"]):
    msg_type = type(msg).__name__
    
    if msg_type == "HumanMessage":
        print(f"[{i}] USER:\n{msg.content[:120]}...\n")
    
    elif msg_type == "AIMessage":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"[{i}] AGENT called tool: '{tc['name']}'")
                print(f"     with args: {tc['args']}\n")
        else:
            print(f"[{i}] AGENT final reply:\n{msg.content[:200]}...\n")
    
    elif msg_type == "ToolMessage":
        print(f"[{i}] TOOL result ('{msg.name}'):\n{msg.content[:200]}...\n")
    
    print("-" * 50)

Full agent reasoning chain:

--------------------------------------------------
[0] USER:

Please analyze my resume against this job description.

MY RESUME:

Skills: Python, FastAPI, PostgreSQL, Docker, Git
Ex...

--------------------------------------------------
[1] AGENT called tool: 'analyze_skill_gap'
     with args: {'job_required_skills': 'Python, Apache Spark, Kafka, Airflow, AWS, Docker, SQL', 'resume_skills': 'Python, FastAPI, PostgreSQL, Docker, Git'}

--------------------------------------------------
[2] TOOL result ('analyze_skill_gap'):
Matched skills  (2): docker, python
Missing skills  (5): airflow, apache spark, aws, kafka, sql
Match score: 29%...

--------------------------------------------------
[3] AGENT final reply:
It looks like you have a good foundation in Python and Docker, but there are some key skills missing from your resume that are required for the Senior Data Engineer role. I would recommend focusing on...

---------------------------------------------

#### Section 6 — Memory

In [19]:
# Add InMemorySaver for multi-turn conversation or add memory so the agent remebers previous turns 
from langgraph.checkpoint.memory import MemorySaver

# 1. Create the memory store 
memory = MemorySaver()

# 2. Create the agent with memory now
agent_with_memory = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=memory         
)

# 3. Create an thread id: this is session id for same chat or conversation 
thread_config = {"configurable": {"thread_id": "user_session_001"}}

print("Agent with memory ready")
print("Thread ID:", thread_config["configurable"]["thread_id"])

Agent with memory ready
Thread ID: user_session_001


C:\Users\raoru.DELL\AppData\Local\Temp\ipykernel_8244\4150615964.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_with_memory = create_react_agent(


In [22]:
# Prove that memory works across multiple turns

def chat(user_input: str) -> str: 
    """Helper to send the message and get the agent's reply"""
    result = agent_with_memory.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config = thread_config
    )
    return result["messages"][-1].content

In [23]:

# Turn 1 — introduce the resume
print("=" * 60)
print("TURN 1")
print("=" * 60)
reply1 = chat(f"Here is my resume:\n{SAMPLE_RESUME}")
print(reply1)

TURN 1
Do you want to refine anything?


In [24]:

# Turn 2 — add the job description (agent should REMEMBER the resume)
print("\n" + "=" * 60)
print("TURN 2")
print("=" * 60)
reply2 = chat(f"Now here is the job I am targeting:\n{SAMPLE_JD}")
print(reply2)


TURN 2
Do you want to refine anything?


In [ ]:

# Turn 3 — follow-up question (no context needed, agent remembers everything)
print("\n" + "=" * 60)
print("TURN 3")
print("=" * 60)
reply3 = chat("Which missing skill should I learn first and why?")
print(reply3)


TURN 3
Based on the job requirements, I recommend learning AWS first. Here's why:

* AWS is a widely used cloud platform in the industry, and having experience with it will make you a more competitive candidate for the Senior Data Engineer role.
* Learning AWS will also help you understand other related technologies, such as Apache Spark and Kafka, which are often used in conjunction with AWS.
* Additionally, AWS has a vast range of services and tools, so learning it will give you a solid foundation for further learning and exploration.

By learning AWS first, you'll be able to build a strong foundation and then move on to learning other missing skills, such as Apache Spark and Kafka.

Do you want to refine anything?


#### Section 7 — Middleware

In [28]:
#  Summarize conversation when it gets too long
from langchain_core.messages import RemoveMessage


def summarize_if_needed(messages: list, max_messages: int = 6) -> list:
    """ 
    If conversation exceeds max_messages, summarize older turns
    and replace them with a single SystemMessage summary.
    This prevents the context window from growing indefinitely.
    """
    if len(messages) <= max_messages:
        return messages
    
    # Separate the last N messages
    recent_messages = messages[-4:]
    older_messages = messages[:-4]

    # Build a summary of the older messages
    older_text = "\n".join(
        f"{type(m).__name__}: {m.content[:100]}"
        for m in older_messages
        if hasattr(m,"content")
    )

    # System prompt
    summary_prompt  = [
        SystemMessage(content="You are a conversation summarizer. Be concise."),
        HumanMessage(content=f"""
                    Summarize this conversation history into 3-4 bullet points.
                    Keep all important facts about the user's resume and target job.

                    HISTORY:
                    {older_text}
                    """)
    ]

    summary_response = llm.invoke(summary_prompt)

    summary_message = SystemMessage(
        content=f"[Conversation summary]\n{summary_response.content}"
    )

    print(f"✅ Summarized {len(older_messages)} older messages into 1 summary")
    return [summary_message] + recent_messages

In [29]:
# Test it
fake_long_history = [
    HumanMessage(content=f"My resume: {SAMPLE_RESUME}"),
    AIMessage(content="Got it. You have Python, FastAPI, PostgreSQL, Docker, Git."),
    HumanMessage(content=f"Target job: {SAMPLE_JD}"),
    AIMessage(content="Missing: Spark, Kafka, Airflow, AWS. Match: 40%."),
    HumanMessage(content="Which skill should I learn first?"),
    AIMessage(content="Learn Apache Spark first — it's the most critical gap."),
    HumanMessage(content="How long does Spark take to learn?"),
]

In [30]:
compressed = summarize_if_needed(fake_long_history, max_messages=6)
print(f"Before: {len(fake_long_history)} messages")
print(f"After : {len(compressed)} messages\n")
print("Summary content:")
print(compressed[0].content)

✅ Summarized 3 older messages into 1 summary
Before: 7 messages
After : 5 messages

Summary content:
[Conversation summary]
Here's a summary of the conversation in 3-4 bullet points:

* The user's resume highlights skills in Python, FastAPI, PostgreSQL, Docker, and Git, with 2 years of experience as a Backend Developer.
* The user is targeting a Senior Data Engineer role.
* The required skills for the target job include Python, Apache Spark, Kafka, Airflow, AWS, and Docker, indicating a need for additional skills beyond the user's current profile.
* The user's current experience and skills may not directly align with the requirements of the Senior Data Engineer role, suggesting a potential gap in qualifications.


#### Section-8 Human-in-the-loop approval

In [31]:
def get_rewrite_with_approval(original_summary: str, job_description: str) -> str:
    """
    Generates a rewritten resume summary, shows it to the human,
    and only applies it after explicit approval.
    This is the human-in-the-loop pattern.
    """
    # Step 1 — agent proposes a rewrite
    rewrite_prompt = [
        SystemMessage(content="You are an expert resume writer."),
        HumanMessage(content=f"""
            Rewrite the following resume summary to better match this job description.
            Return ONLY the rewritten summary — no explanations.

            ORIGINAL SUMMARY:
            {original_summary}

            JOB DESCRIPTION:
            {job_description}
            """)
            ]

    proposed_rewrite = llm.invoke(rewrite_prompt).content

    # Step 2 — show the proposal to the human
    print("=" * 60)
    print("ORIGINAL:")
    print(original_summary.strip())
    print("\nPROPOSED REWRITE:")
    print(proposed_rewrite.strip())
    print("=" * 60)

    # Step 3 — wait for explicit human approval
    approval = input("\nApprove this rewrite? (yes / no / edit): ").strip().lower()

    if approval == "yes":
        print("Rewrite approved and applied.")
        return proposed_rewrite

    elif approval == "edit":
        edited = input("Enter your edited version: ").strip()
        print("Edited version saved.")
        return edited

    else:
        print("Rewrite rejected. Original kept.")
        return original_summary


# Test it
original = "Experienced developer with good technical skills."
final_summary = get_rewrite_with_approval(original, SAMPLE_JD)

print("\nFinal summary that will be used:")
print(final_summary)

ORIGINAL:
Experienced developer with good technical skills.

PROPOSED REWRITE:
Highly skilled Senior Data Engineer with expertise in designing, developing, and deploying scalable data pipelines using Python, Apache Spark, Kafka, Airflow, and AWS, with a strong foundation in containerization using Docker and SQL.
Rewrite approved and applied.

Final summary that will be used:
Highly skilled Senior Data Engineer with expertise in designing, developing, and deploying scalable data pipelines using Python, Apache Spark, Kafka, Airflow, and AWS, with a strong foundation in containerization using Docker and SQL.
